In [ ]:
import pandas as pd
import numpy as np
import joblib

# Load your trained model and scaler from repo models
model = joblib.load('../src/models/xgb_procrastination_model.pkl')
scaler = joblib.load('../src/models/scaler.pkl')

# Use the project's trained feature set
feature_cols = [
    'last_minute_ratio',
    'engagement_intensity',
    'deadline_pressure',
    'login_consistency',
    'early_starter',
    'completion_rate',
    'activity_span'
]

NUDGE_THRESHOLD = 0.5

dynamic_results = []

for student_id, group in test_df.groupby('student_id'):
    group = group.sort_values('week_number')
    nudge_sent = False
    detected_early = False  # flagged > 7 days before deadline
    false_alarm = False

    for _, row in group.iterrows():
        features = scaler.transform([row[feature_cols].values])
        risk_score = model.predict_proba(features)[0][1]

        if risk_score >= NUDGE_THRESHOLD and not nudge_sent:
            nudge_sent = True
            # Early detection = nudge fired with > 7 days remaining
            if row['days_before_deadline'] > 7:
                detected_early = True
            # False alarm = nudge fired but student actually submitted on time
            if row['actual_outcome'] == 0:
                false_alarm = True

    dynamic_results.append({
        'student_id': student_id,
        'nudge_sent': nudge_sent,
        'detected_early': detected_early,
        'false_alarm': false_alarm,
        'actual_outcome': group['actual_outcome'].max()
    })

dynamic_df = pd.DataFrame(dynamic_results)

In [ ]:
static_results = []

for student_id, group in test_df.groupby('student_id'):
    group = group.sort_values('week_number')
    weeks = group['week_number'].tolist()
    actual_outcome = group['actual_outcome'].max()

    # Static: nudge fires every 7th day (weeks 1, 2, 3...)
    # We treat each week as 7 days, so nudge fires every week
    nudge_sent = len(weeks) > 0  # always fires at least once
    
    # For static, first nudge is at week 1
    first_nudge_week = weeks[0] if weeks else None
    days_remaining_at_nudge = group.loc[
        group['week_number'] == first_nudge_week, 'days_before_deadline'
    ].values[0] if first_nudge_week else 0

    detected_early = days_remaining_at_nudge > 7
    false_alarm = (actual_outcome == 0)  # nudge fired but student was fine

    static_results.append({
        'student_id': student_id,
        'nudge_sent': nudge_sent,
        'detected_early': detected_early,
        'false_alarm': false_alarm,
        'actual_outcome': actual_outcome
    })

static_df = pd.DataFrame(static_results)

In [ ]:
from scipy.stats import wilcoxon

def compute_metrics(df):
    at_risk = df[df['actual_outcome'] == 1]
    
    # Early detection rate: of truly at-risk students, 
    # how many were detected > 7 days before deadline
    early_detection_rate = at_risk['detected_early'].mean() * 100

    # False alarm rate: of ALL nudges sent, 
    # how many went to students who submitted on time
    nudged = df[df['nudge_sent'] == True]
    false_alarm_rate = nudged['false_alarm'].mean() * 100

    # Nudge efficiency: true positives / total nudges sent
    true_positives = nudged[nudged['actual_outcome'] == 1].shape[0]
    total_nudges = nudged.shape[0]
    efficiency = true_positives / total_nudges if total_nudges > 0 else 0

    return early_detection_rate, false_alarm_rate, efficiency

dyn_edr, dyn_far, dyn_eff = compute_metrics(dynamic_df)
sta_edr, sta_far, sta_eff = compute_metrics(static_df)

print(f"Early Detection — Dynamic: {dyn_edr:.1f}%  Static: {sta_edr:.1f}%")
print(f"False Alarm    — Dynamic: {dyn_far:.1f}%  Static: {sta_far:.1f}%")
print(f"Efficiency     — Dynamic: {dyn_eff:.2f}   Static: {sta_eff:.2f}")

# Statistical significance test
stat, p = wilcoxon(dynamic_df['detected_early'], static_df['detected_early'])
print(f"Wilcoxon p-value: {p:.4f}")

In [ ]:
# How many days before deadline was the first nudge triggered?

def mean_lead_time(df, test_df_grouped):
    lead_times = []
    for student_id, row in df[df['nudge_sent'] == True].iterrows():
        group = test_df_grouped.get_group(row['student_id'])
        # get days_before_deadline at the row where nudge was first sent
        # (already captured in your loop above — store it)
        pass  # add days_before_deadline_at_nudge to your results dict above

# Simpler: just add 'days_at_nudge' to your results dict in Steps 2 & 3
# then: dynamic_df['days_at_nudge'].mean()